In [31]:
# 기본 라이브러리
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

In [32]:
# 1. 7종 생선 LogisticRegression — 확률 예측 보고서


fish = pd.read_csv("https://raw.githubusercontent.com/rickiepark/hg-mldl/master/fish.csv")

# 종별 샘플 수
print(fish["Species"].value_counts())

# StandardScaler
ss = StandardScaler()
train_scaled = ss.fit_transform(train_input)
test_scaled = ss.transform(test_input)

# 모델 학습
lr = LogisticRegression(C=20, max_iter=1000)
lr.fit(train_scaled, train_target)

# classes 출력
print("classes:", lr.classes_)

# predict_proba → DataFrame
proba = lr.predict_proba(test_scaled[:5])

proba_df = pd.DataFrame(
    np.round(proba, 3),
    columns=lr.classes_
)

display(proba_df)

# train/test 정확도
print("train accuracy:", lr.score(train_scaled, train_target))
print("test accuracy:", lr.score(test_scaled, test_target))

# 예측값 / 실제값
pred = lr.predict(test_scaled[:5])

print("예측값:", pred)
print("실제값:", test_target[:5])

Species
Perch        56
Bream        35
Roach        20
Pike         17
Smelt        14
Parkki       11
Whitefish     6
Name: count, dtype: int64
classes: ['Bream' 'Parkki' 'Perch' 'Pike' 'Roach' 'Smelt' 'Whitefish']


,Bream,Parkki,Perch,Pike,Roach,Smelt,Whitefish
0,0.000,0.014,0.842,0.000,0.135,0.007,0.003
1,0.000,0.003,0.044,0.000,0.007,0.946,0.000
2,0.000,0.000,0.034,0.934,0.015,0.016,0.000
3,0.011,0.034,0.305,0.006,0.567,0.000,0.076
4,0.000,0.000,0.904,0.002,0.089,0.002,0.001


train accuracy: 0.9327731092436975
test accuracy: 0.925
예측값: ['Perch' 'Smelt' 'Pike' 'Roach' 'Perch']
실제값: ['Perch' 'Smelt' 'Pike' 'Whitefish' 'Perch']


In [33]:
# 2. 타이타닉 불균형 분류 — Confusion Matrix + 평가 지표 보고서


# titanic dropna → 714개 확인
titanic = sns.load_dataset("titanic")
titanic = titanic.dropna()

print("dropna 후 데이터 개수:", len(titanic))

X = titanic[["pclass", "age", "sibsp", "parch", "fare"]]
y = titanic["survived"]

# stratify=y 옵션으로 불균형 비율 유지
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

# classification_report target_names 설정
print("Classification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Dead", "Survived"]
))

# AUC 계산 시 predict_proba[:, 1] 사용
y_proba = model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_proba))

# 무조건 사망 예측 정확도 직접 계산
always_dead = np.zeros(len(y_test))

print(
    "무조건 사망 예측 정확도:",
    accuracy_score(y_test, always_dead)
)

dropna 후 데이터 개수: 182
Confusion Matrix
[[ 5 10]
 [ 2 29]]
Classification Report
              precision    recall  f1-score   support

        Dead       0.71      0.33      0.45        15
    Survived       0.74      0.94      0.83        31

    accuracy                           0.74        46
   macro avg       0.73      0.63      0.64        46
weighted avg       0.73      0.74      0.71        46

ROC-AUC: 0.6946236559139785
무조건 사망 예측 정확도: 0.32608695652173914


In [34]:
# 3. 유방암 데이터셋 — 의료 분류 + FN 최소화 전략


from sklearn.datasets import load_breast_cancer

# data.target_names 출력으로 label 방향 확인
data = load_breast_cancer()

print("target_names:", data.target_names)
# 원래 기준: 0 = malignant(악성), 1 = benign(양성)

# y = 1 - data.target 적용
# 악성을 1로 만들기 위해 label 방향을 뒤집음
X = data.data
y = 1 - data.target

print("0 = 양성, 1 = 악성")
print(pd.Series(y).value_counts())

# stratify=y 분할
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# 기본 기준 0.5
y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred_05 = (y_prob >= 0.5).astype(int)

print("Threshold 0.5 결과")
print(confusion_matrix(y_test, y_pred_05))
print(classification_report(
    y_test,
    y_pred_05,
    target_names=["Benign", "Malignant"]
))

# y_pred_custom = (y_prob >= 0.3).astype(int) 적용
y_pred_custom = (y_prob >= 0.3).astype(int)

print("Threshold 0.3 결과")
print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(
    y_test,
    y_pred_custom,
    target_names=["Benign", "Malignant"]
))

# Recall 변화 이유 1~2문장 설명
print("""
Threshold를 0.3으로 낮추면 악성으로 판단하는 기준이 완화된다.
그래서 FN이 감소하고 Recall이 증가한다.
""")

target_names: ['malignant' 'benign']
0 = 양성, 1 = 악성
0    357
1    212
Name: count, dtype: int64
Threshold 0.5 결과
[[89  1]
 [ 4 49]]
              precision    recall  f1-score   support

      Benign       0.96      0.99      0.97        90
   Malignant       0.98      0.92      0.95        53

    accuracy                           0.97       143
   macro avg       0.97      0.96      0.96       143
weighted avg       0.97      0.97      0.96       143

Threshold 0.3 결과
[[89  1]
 [ 2 51]]
              precision    recall  f1-score   support

      Benign       0.98      0.99      0.98        90
   Malignant       0.98      0.96      0.97        53

    accuracy                           0.98       143
   macro avg       0.98      0.98      0.98       143
weighted avg       0.98      0.98      0.98       143


Threshold를 0.3으로 낮추면 악성으로 판단하는 기준이 완화된다.
그래서 FN이 감소하고 Recall이 증가한다.

